# 03 · Cuarentena, decisión y revalidación

**Objetivo:** mostrar cómo corregir sin borrar evidencia ni contar dos veces el mismo registro.

**Modalidad:** material para explicar en clase. Los ejemplos se ejecutan únicamente en memoria, con datos ficticios. No conectan a una base, no crean archivos ni ejecutan una ingesta. Los bloques SQL son ilustrativos y no se envían a ningún servidor.

**Prerrequisitos:** Python con pandas. Cada notebook es independiente: no requiere scripts `.py` ni haber ejecutado otra notebook.


In [ ]:
# Importar conexión centralizada desde el módulo
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO, USUARIO

# Obtener los engines
engine_origen, engine_destino = obtener_engines()
print(f'Conexiones establecidas correctamente.')
print(f'Servidor: {SERVIDOR}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')


## El rechazo tiene un motivo y un responsable

```text
Detectado -> Cuarentena -> Steward revisa -> Corrección auditada
                                          -> Revalidación -> Aceptado
```

Si la corrección sigue incumpliendo, vuelve a cuarentena. No existe una
aprobación implícita por haber editado un campo.


In [ ]:
from copy import deepcopy
from decimal import Decimal, InvalidOperation
import pandas as pd

raw = {"id": 1, "producto": 999, "total": "ERROR"}

def motivos(registro):
    problemas = []
    if registro["producto"] not in {10, 20}:
        problemas.append("Q_REF_PRODUCTO")
    try:
        numero = Decimal(registro["total"])
        if not numero.is_finite() or numero != numero.quantize(Decimal("0.01")):
            problemas.append("Q_DECIMAL_TOTAL")
    except (InvalidOperation, ValueError):
        problemas.append("Q_DECIMAL_TOTAL")
    return problemas

cuarentena = [{"lote": "CLASE-001", "id": raw["id"], "motivos": motivos(raw)}]
display(pd.DataFrame(cuarentena))
print("Registros:", len(cuarentena), "Motivos:", len(cuarentena[0]["motivos"]))


In [ ]:
correccion = {
    "actor": "steward_demo", "motivo": "Restaurar valores de la ficha sintética",
    "antes": deepcopy(raw),
    "cambios": {"producto": 10, "total": "24.20"},
}
corregido = {**raw, **correccion["cambios"]}
correccion["despues"] = deepcopy(corregido)
pendientes = motivos(corregido)
display(pd.DataFrame([correccion]))
print("Motivos tras revalidación:", pendientes)
print("RAW original:", raw)
print("¿Aprobado?:", len(pendientes) == 0)


## Qué conservaría un sistema real

Lote, tabla, PK, versión de reglas, motivos, fecha, actor, justificación, antes
y después. Una fila puede incumplir muchas reglas: el total de cuarentena se
calcula por registro, no por motivo.

No corregimos la DB de origen desde esta práctica. El ejemplo usa un diccionario
ficticio y conserva `raw` intacto. En producción debe decidirse si el arreglo
corresponde al origen o a un tratamiento gobernado en la capa de transformación.


## Actividad de discusión

¿Alcanza con arreglar el Decimal? **No:** mientras producto=999, la integridad sigue fallando. Revalidar todas las reglas evita aprobarlo.